# TinyML Lab 3 — SLM results collector

This notebook converts the executable Python examples from `main.html` into repeatable experiments for the lab questions. It assumes Ollama is running locally on the Raspberry Pi. Run the setup cells once, then rerun only the experiment cells you need.

**Important:** the HTML handout asks you to record memory and CPU temperature separately with `htop`, `free -h`, and `vcgencmd`. The notebook records the Ollama timings automatically and leaves fields for those system measurements.

## 0. Configuration

Edit paths and prompts here before starting. Put your three test images in `IMAGE_DIR`.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, time, platform, subprocess
import pandas as pd

MODEL_NAMES = ['llama3.2:1b', 'gemma2:2b', 'phi3.5:3.8b']
BENCHMARK_PROMPT = 'What is the capital of France? Explain briefly.'
IMAGE_MODEL = 'llava-phi3:3.8b'
RAG_MODEL = 'llama3.2:3b'
EMBEDDING_MODEL = 'nomic-embed-text'
IMAGE_DIR = Path('images')
RESULTS_DIR = Path('lab3_results')
RESULTS_DIR.mkdir(exist_ok=True)
print('Working directory:', Path.cwd())
print('Images directory:', IMAGE_DIR.resolve())

In [ ]:
# Run this only if the packages are not installed in the active environment.
%pip install -q ollama pandas pillow matplotlib chromadb haversine pydantic openai instructor

In [ ]:
import ollama

print('Installed Ollama models:')
print(ollama.list())

# Pull models once from a terminal if they are missing:
# ollama pull llama3.2:1b
# ollama pull gemma2:2b
# ollama pull phi3.5:3.8b
# ollama pull llava-phi3:3.8b
# ollama pull llama3.2:3b
# ollama pull nomic-embed-text

## 1. Model performance comparison

The notebook captures the same metrics exposed by `ollama.generate()`. Add peak memory and CPU temperature from the monitoring commands after each run.

In [ ]:
def _rate(count, duration_ns):
    return count / (duration_ns / 1e9) if duration_ns else None

def benchmark_model(model, prompt=BENCHMARK_PROMPT, temperature=0.0):
    started = datetime.now().isoformat(timespec='seconds')
    response = ollama.generate(model=model, prompt=prompt, options={'temperature': temperature})
    row = {
        'timestamp': started, 'model': model, 'prompt': prompt,
        'response': response.get('response', ''),
        'total_duration_s': response.get('total_duration', 0) / 1e9,
        'load_duration_s': response.get('load_duration', 0) / 1e9,
        'prompt_eval_count': response.get('prompt_eval_count'),
        'prompt_eval_rate_tokens_s': _rate(response.get('prompt_eval_count', 0), response.get('prompt_eval_duration', 0)),
        'eval_count': response.get('eval_count'),
        'eval_rate_tokens_s': _rate(response.get('eval_count', 0), response.get('eval_duration', 0)),
        'peak_memory_mb_manual': None, 'peak_cpu_temp_c_manual': None,
    }
    return row

benchmark_rows = []
for model in MODEL_NAMES:
    print(f'Running {model}...')
    row = benchmark_model(model)
    benchmark_rows.append(row)
    print(f"{row['total_duration_s']:.2f}s | prompt {row['prompt_eval_rate_tokens_s']:.1f} tok/s | eval {row['eval_rate_tokens_s']:.1f} tok/s")

benchmark_df = pd.DataFrame(benchmark_rows)
display(benchmark_df[['model', 'total_duration_s', 'prompt_eval_rate_tokens_s', 'eval_rate_tokens_s', 'peak_memory_mb_manual', 'peak_cpu_temp_c_manual']])
benchmark_df.to_csv(RESULTS_DIR / 'model_benchmark.csv', index=False)
benchmark_df.to_json(RESULTS_DIR / 'model_benchmark.json', orient='records', indent=2)

### Manual Raspberry Pi monitoring

During a benchmark, use separate terminals:

```bash
watch -n 1 vcgencmd measure_temp
watch -n 1 free -h
htop
```

Then fill `peak_memory_mb_manual` and `peak_cpu_temp_c_manual` in the dataframe and rerun the save cell.

In [ ]:
# Optional: quick table for the written comparison.
fastest = benchmark_df.loc[benchmark_df['total_duration_s'].idxmin(), 'model']
slowest = benchmark_df.loc[benchmark_df['total_duration_s'].idxmax(), 'model']
print('Fastest by total duration:', fastest)
print('Slowest by total duration:', slowest)
display(benchmark_df.sort_values('eval_rate_tokens_s', ascending=False))

## 2. Multimodal image analysis

Add at least three images (ideally cityscape, nature, indoor) to `images/`. The first run may include model-load time, so compare it with later runs.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def analyze_image(image_path, prompt='Describe this image carefully. Separate observations from guesses.'):
    image_path = Path(image_path)
    started = time.perf_counter()
    with image_path.open('rb') as f:
        response = ollama.generate(model=IMAGE_MODEL, prompt=prompt, images=[f.read()], options={'temperature': 0})
    elapsed = time.perf_counter() - started
    return {
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'image': str(image_path), 'wall_time_s': elapsed,
        'ollama_total_duration_s': response.get('total_duration', 0) / 1e9,
        'description': response.get('response', ''),
        'hallucinations_or_errors': '',
        'cpu_temp_before_c': None, 'cpu_temp_peak_c': None, 'cpu_temp_after_c': None,
    }

image_paths = sorted([p for p in IMAGE_DIR.glob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}])
print('Found images:', image_paths)
image_rows = []
for path in image_paths[:3]:
    display(Image.open(path).resize((400, 300)))
    row = analyze_image(path)
    image_rows.append(row)
    print(path.name, f"{row['wall_time_s']:.2f}s")
    print(row['description'])

image_df = pd.DataFrame(image_rows)
image_df.to_csv(RESULTS_DIR / 'image_analysis.csv', index=False)
display(image_df[['image', 'wall_time_s', 'ollama_total_duration_s', 'hallucinations_or_errors']])

## 3. Structured function calling: capital coordinates and distance

This uses the handout's OpenAI-compatible Ollama endpoint and Pydantic schema. The original HTML has nested double quotes in two f-strings; this notebook uses safe quoting.

In [ ]:
from pydantic import BaseModel, Field
from haversine import haversine
from openai import OpenAI
import instructor

class CityCoord(BaseModel):
    city: str = Field(description='Name of the capital city')
    lat: float = Field(description='Decimal latitude')
    lon: float = Field(description='Decimal longitude')

structured_client = instructor.patch(
    OpenAI(base_url='http://localhost:11434/v1', api_key='ollama'),
    mode=instructor.Mode.JSON,
)
SANTIAGO = (-33.33, -70.51)

def get_capital_coordinates(country, model='phi3.5:3.8b'):
    return structured_client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': f'Return the decimal latitude and decimal longitude of the capital of {country}.'}],
        response_model=CityCoord, max_retries=10, temperature=0,
    )

countries = ['France', 'Italy', 'Brazil', 'Japan', 'South Africa']
distance_rows = []
for country in countries:
    try:
        coord = get_capital_coordinates(country)
        distance_rows.append({'country': country, 'capital': coord.city, 'lat': coord.lat, 'lon': coord.lon, 'distance_from_santiago_km': haversine(SANTIAGO, (coord.lat, coord.lon)), 'error_percent': None})
    except Exception as exc:
        distance_rows.append({'country': country, 'capital': None, 'lat': None, 'lon': None, 'distance_from_santiago_km': None, 'error_percent': None, 'error': repr(exc)})

distance_df = pd.DataFrame(distance_rows)
distance_df.to_csv(RESULTS_DIR / 'capital_distances.csv', index=False)
display(distance_df)

For the error percentage, enter the ground-truth distances you look up in the `actual_distance_km` column, then run the next cell. The formula is `abs(calculated - actual) / actual * 100`.

In [ ]:
# Fill these values from your chosen ground-truth source.
distance_df['actual_distance_km'] = None
distance_df['error_percent'] = (
    (distance_df['distance_from_santiago_km'] - distance_df['actual_distance_km']).abs()
    / distance_df['actual_distance_km'] * 100
)
distance_df.to_csv(RESULTS_DIR / 'capital_distances.csv', index=False)
display(distance_df)

## 4. RAG: base model versus retrieved context

In [ ]:
import chromadb

bee_documents = [
    'Bee-keeping, also known as apiculture, involves maintaining bee colonies.',
    'The European honey bee is Apis mellifera.',
    'A typical bee colony contains between 20,000 and 80,000 bees.',
    'A queen bee can lay up to 2,000 eggs per day during peak seasons.',
    'Common pests and diseases include varroa mites and foulbrood.',
    'Worker bees collect nectar and pollen and care for the brood.',
    'Honey bees communicate the location of food using the waggle dance.',
    'Bees pollinate many flowering plants and agricultural crops.',
    'A queen is normally the primary reproductive female in a colony.',
    'Drone bees are male bees whose main role is mating with a queen.',
    'Bees can be affected by habitat loss, pesticides, parasites, and disease.',
    'Honey is made from nectar collected by foraging bees.',
    'Bees regulate hive temperature by clustering and ventilating the hive.',
    'Brazil has native stingless bees as well as introduced honey bees.',
    'Varroa mites feed on developing and adult bees and can spread viruses.',
]

rag_client = chromadb.Client()
collection = rag_client.get_or_create_collection(name='lab3_bee_facts')
for i, doc in enumerate(bee_documents):
    emb = ollama.embeddings(model=EMBEDDING_MODEL, prompt=doc)['embedding']
    collection.upsert(ids=[str(i)], embeddings=[emb], documents=[doc])

def answer_base(question):
    return ollama.generate(model=RAG_MODEL, prompt=question, options={'temperature': 0})['response']

def answer_rag(question, n_results=5):
    query_embedding = ollama.embeddings(model=EMBEDDING_MODEL, prompt=question)['embedding']
    docs = collection.query(query_embeddings=[query_embedding], n_results=n_results)['documents'][0]
    context = '\n'.join(f'- {doc}' for doc in docs)
    prompt = f'Answer only from the context below. If the context is insufficient, say so.\n\nContext:\n{context}\n\nQuestion: {question}'
    return ollama.generate(model=RAG_MODEL, prompt=prompt, options={'temperature': 0})['response'], docs

questions = ['How many eggs does a queen bee lay?', 'What pests affect bees?', 'Are there bees in Brazil?']
rag_rows = []
for question in questions:
    base = answer_base(question)
    rag, sources = answer_rag(question)
    rag_rows.append({'question': question, 'base_response': base, 'rag_response': rag, 'retrieved_documents': ' | '.join(sources)})
    print('QUESTION:', question, '\nBASE:', base, '\nRAG:', rag, '\n')

rag_df = pd.DataFrame(rag_rows)
rag_df.to_csv(RESULTS_DIR / 'rag_comparison.csv', index=False)
display(rag_df[['question', 'base_response', 'rag_response']])

## 5. Save a single experiment bundle

Run this at the end. The CSV files in `lab3_results/` are ready to use in your report.

In [ ]:
bundle = {
    'created_at': datetime.now().isoformat(),
    'platform': platform.platform(),
    'python': platform.python_version(),
    'benchmark_prompt': BENCHMARK_PROMPT,
    'models': MODEL_NAMES,
    'files': sorted(p.name for p in RESULTS_DIR.glob('*')),
}
(RESULTS_DIR / 'experiment_metadata.json').write_text(json.dumps(bundle, indent=2), encoding='utf-8')
print(json.dumps(bundle, indent=2))
print('Saved results to:', RESULTS_DIR.resolve())